In [19]:
import pandas as pd
import os
import csv  # 🚨 Added this import to access the quoting rules

def split_excel_to_csv():
    input_excel = "../data/Gen_AI Dataset.xlsx"
    train_output = "../data/train.csv"
    test_output = "../data/test.csv"
    
    print(f"📂 Looking for {input_excel}...")
    
    if not os.path.exists(input_excel):
        print(f"❌ Error: Could not find '{input_excel}'. Please ensure it is in the data/ folder.")
        return

    try:
        sheets = pd.read_excel(input_excel, sheet_name=None)
        sheet_names = list(sheets.keys())
        
        print(f"✅ Found {len(sheet_names)} tabs: {sheet_names}")
        
        train_sheet_name = next((name for name in sheet_names if 'train' in name.lower()), None)
        test_sheet_name = next((name for name in sheet_names if 'test' in name.lower()), None)
        
        if train_sheet_name:
            print(f"💾 Saving '{train_sheet_name}' to {train_output} with quotes...")
            # 🚨 Added quoting=csv.QUOTE_ALL here
            sheets[train_sheet_name].to_csv(train_output, index=False, quoting=csv.QUOTE_ALL)
        else:
            print("⚠️ Could not find a tab with 'train' in the name.")
            
        if test_sheet_name:
            print(f"💾 Saving '{test_sheet_name}' to {test_output} with quotes...")
            # 🚨 Added quoting=csv.QUOTE_ALL here
            sheets[test_sheet_name].to_csv(test_output, index=False, quoting=csv.QUOTE_ALL)
        else:
            print("⚠️ Could not find a tab with 'test' in the name.")
            
        print("\n🎉 Split complete! Your CSVs are strictly enclosed by quotes.")
        
    except Exception as e:
        print(f"❌ Failed to process the Excel file: {e}")

if __name__ == "__main__":
    split_excel_to_csv()

📂 Looking for ../data/Gen_AI Dataset.xlsx...
✅ Found 2 tabs: ['Train-Set', 'Test-Set']
💾 Saving 'Train-Set' to ../data/train.csv with quotes...
💾 Saving 'Test-Set' to ../data/test.csv with quotes...

🎉 Split complete! Your CSVs are strictly enclosed by quotes.


# Evaluation on Train Set

In [20]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

# Normalize column names 
train_df.columns = [col.strip().lower() for col in train_df.columns]
query_col = next((col for col in train_df.columns if 'query' in col), train_df.columns[0])
url_col = next((col for col in train_df.columns if 'url' in col), train_df.columns[1])

# Group the expected URLs by query
ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [str(url).strip().rstrip('/') for url in x if pd.notna(url)]
).to_dict()

print(f"✅ Found {len(ground_truth)} unique queries to evaluate.\n")

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_urls in ground_truth.items():
    try:
        # Send the POST request to your running API
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # Extract the list from the exact key your API uses
            predictions = json_data.get("recommended_assessments", [])
            
            # Extract URLs from the JSON response
            predicted_urls = [res.get("url", "").strip().rstrip('/') for res in predictions]
            
            # Calculate hits
            hits = sum(1 for url in expected_urls if url in predicted_urls)
            total_expected = len(expected_urls)
            
            # Calculate Recall@10
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                
                print(f"Query: '{query[:60]}...'")
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}\n")
                
                results_log.append({
                    "Query": query,
                    "Expected Count": total_expected,
                    "Hits": hits,
                    "Recall@10": recall,
                    "Missed URLs": [url for url in expected_urls if url not in predicted_urls]
                })
            else:
                print(f"⚠️ No expected URLs found for query: '{query[:40]}...'")
                
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            print(response.text)
            
        # Brief pause to avoid overwhelming your local server / Groq API rate limits
        time.sleep(1) 
        
    except requests.exceptions.ConnectionError:
        print("🚨 CRITICAL: Could not connect to the API. Is `python api.py` running?")
        break
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)
else:
    print("⚠️ Could not calculate Mean Recall.")

📂 Loading ground truth from ../data/train.csv...
✅ Found 10 unique queries to evaluate.

🚀 Hitting the FastAPI Endpoint for Evaluation...

🚨 CRITICAL: Could not connect to the API. Is `python api.py` running?
⚠️ Could not calculate Mean Recall.


In [ ]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    # Get the very last part of the URL path (e.g., 'java-8-new')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

# Find the exact names of the Query and URL columns based on your sample
query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

# Group the expected URLs by query and apply the slug extractor
ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        # Ask your API for 10 recommendations (for Recall@10)
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            predictions = json_data.get("recommended_assessments", [])
            
            # Apply the slug extractor to your API's predicted URLs
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # Calculate hits based ONLY on the slug!
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                
                print(f"Query: '{query[:60]}...'")
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}\n")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
            
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...

Query: 'Based on the JD below recommend me assessment for the Consul...'
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Find me 1 hour long assesment for the below job at SHL
Job D...'
  ✅ Hits: 0/9 -> Recall@10: 0.00

Query: 'I am hiring for Java developers who can also collaborate eff...'
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'I am looking for a COO for my company in China and I want to...'
  ✅ Hits: 0/6 -> Recall@10: 0.00

Query: 'I want to hire a Senior Data Analyst with 5 years of experie...'
  ✅ Hits: 0/10 -> Recall@10: 0.00

Query: 'I want to hire new graduates for a sales role in my company,...'
  ✅ Hits: 0/9 -> Recall@10: 0.00

Query: 'ICICI Bank Assistant Admin, Experience required 0-2 years, t...'
  ✅ Hits: 0/6 -> Recall@10: 0.00

Query: 'KEY RESPONSIBITILES:

Manage the sound-scape 

In [ ]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (0): []
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-opq32r', 'search-engine-optimization-new']
  🔍 Predicted Slugs (10): ['written-english-v1', 'reading-comprehension-english-v1', 'reading-comprehension-v2', 'written-spanish', 'proofreading-v1', 'reading-comprehension-spanish-v1', 'writex-email-writing-managerial-new', 'english-comprehension-new', 'verify-verbal-ability-next-generatio

In [ ]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (0): []
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-opq32r', 'search-engine-optimization-new']
  🔍 Predicted Slugs (10): ['written-english-v1', 'reading-comprehension-english-v1', 'reading-comprehension-v2', 'written-spanish', 'proofreading-v1', 'reading-comprehension-spanish-v1', 'writex-email-writing-managerial-new', 'english-comprehension-new', 'verify-verbal-ability-next-generatio

In [ ]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (0): []
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-opq32r', 'search-engine-optimization-new']
  🔍 Predicted Slugs (10): ['written-english-v1', 'reading-comprehension-english-v1', 'reading-comprehension-v2', 'written-spanish', 'proofreading-v1', 'reading-comprehension-spanish-v1', 'writex-email-writing-managerial-new', 'english-comprehension-new', 'verify-verbal-ability-next-generatio

# After Prompt Change

In [ ]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (10): ['software-business-analysis', 'pjm-development-report', 'interviewing-and-hiring-concepts-u-s', 'pjm-selection-report', 'human-resources-new', 'opq-ucf-development-action-planner-report', 'sql-server-analysis-services-%28ssas%29-%28new%29', 'virtual-assessment-and-development-centers', 'training-development', 'entry-level-sales-solution']
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionn

In [21]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (10): ['software-business-analysis', 'pjm-selection-report', 'sql-server-analysis-services-%28ssas%29-%28new%29', 'opq-ucf-development-action-planner-report', 'human-resources-new', 'hipo-assessment-report-1-0', 'sap-basis-new', 'hipo-assessment-report-2-0', 'job-control-language-new', 'pjm-development-report']
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-opq32r', 'search-engine-optimi

In [22]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (10): ['human-resources-new', 'opq-ucf-development-action-planner-report', 'job-control-language-new', 'pjm-selection-report', 'workplace-administration-skills-new', 'pjm-development-report', 'project-management-2013', 'hipo-assessment-report-1-0', 'business-communications', 'opq-manager-plus-report']
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-opq32r', 'search-engine-optimization-new

In [23]:
import pandas as pd
import requests
import time

# --- Configuration ---
API_URL = "http://127.0.0.1:8000/recommend"
TRAIN_PATH = "../data/train.csv"

# --- URL Normalization Helper ---
def get_slug(url):
    """Extracts just the final test name from the URL to bypass domain/path mismatches."""
    url = str(url).strip().rstrip('/')
    slug = url.split('/')[-1]
    return slug.lower()

# --- 1. Load Ground Truth ---
print(f"📂 Loading ground truth from {TRAIN_PATH}...")
train_df = pd.read_csv(TRAIN_PATH)

query_col = "Query" if "Query" in train_df.columns else train_df.columns[0]
url_col = "Assessment_url" if "Assessment_url" in train_df.columns else train_df.columns[1]

ground_truth = train_df.groupby(query_col)[url_col].apply(
    lambda x: [get_slug(url) for url in x if pd.notna(url)]
).to_dict()

# --- 2. Evaluation Loop ---
recalls = []
results_log = []

print("🚀 Hitting the FastAPI Endpoint for Evaluation...\n")

for query, expected_slugs in ground_truth.items():
    try:
        response = requests.post(API_URL, json={"query": query})
        
        if response.status_code == 200:
            json_data = response.json()
            
            # 🚨 BUG FIX: Catch the list no matter what the API called it
            if isinstance(json_data, list):
                predictions = json_data
            elif isinstance(json_data, dict):
                predictions = json_data.get("recommended_assessments") or json_data.get("recommended_assignments") or []
            else:
                predictions = []
                
            predicted_slugs = [get_slug(res.get("url", "")) for res in predictions]
            
            # 🚨 DEBUG LOGS: Let's see exactly what we are comparing!
            print(f"\nQuery: '{query[:60]}...'")
            print(f"  🔍 Expected Slugs  ({len(expected_slugs)}): {expected_slugs}")
            print(f"  🔍 Predicted Slugs ({len(predicted_slugs)}): {predicted_slugs}")
            
            hits = sum(1 for slug in expected_slugs if slug in predicted_slugs)
            total_expected = len(expected_slugs)
            
            if total_expected > 0:
                recall = hits / total_expected
                recalls.append(recall)
                print(f"  ✅ Hits: {hits}/{total_expected} -> Recall@10: {recall:.2f}")
                
                results_log.append({
                    "Query": query,
                    "Expected": expected_slugs,
                    "Predicted": predicted_slugs,
                    "Recall@10": recall
                })
        else:
            print(f"❌ API Error {response.status_code} for query: '{query[:40]}...'")
            
        time.sleep(1) 
        
    except Exception as e:
        print(f"❌ Error evaluating query: '{query[:40]}...'\nError: {e}")

# --- 3. Final Score Calculation ---
if recalls:
    mean_recall = sum(recalls) / len(recalls)
    print("\n" + "="*50)
    print(f"🏆 FINAL SCORE: Mean Recall@10 = {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print("="*50)

📂 Loading ground truth from ../data/train.csv...
🚀 Hitting the FastAPI Endpoint for Evaluation...


Query: 'Based on the JD below recommend me assessment for the Consul...'
  🔍 Expected Slugs  (5): ['shl-verify-interactive-numerical-calculation', 'administrative-professional-short-form', 'verify-verbal-ability-next-generation', 'occupational-personality-questionnaire-opq32r', 'professional-7-1-solution']
  🔍 Predicted Slugs (10): ['software-business-analysis', 'pjm-selection-report', 'sql-server-analysis-services-%28ssas%29-%28new%29', 'pjm-development-report', 'project-management-2013', 'opq-ucf-development-action-planner-report', 'human-resources-new', 'hipo-assessment-report-1-0', 'agile-software-development', 'virtual-assessment-and-development-centers']
  ✅ Hits: 0/5 -> Recall@10: 0.00

Query: 'Content Writer required, expert in English and SEO....'
  🔍 Expected Slugs  (5): ['english-comprehension-new', 'drupal-new', 'written-english-v1', 'occupational-personality-questionnaire-op